In [1]:
# Bootstrap the repository before importing project modules.
from pathlib import Path
import sys

_NOTEBOOK_START = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (_NOTEBOOK_START, *_NOTEBOOK_START.parents)
     if (candidate / "src" / "evaluation.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root containing src/evaluation.py.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import evaluate_predictions, build_model_comparison, calculate_runtime_metrics
from src.models import calculate_anomaly_scores

print("Project root:", PROJECT_ROOT)
print("Evaluation and model modules imported successfully.")


Project root: D:\projects\server-performance-anomaly-detection
Evaluation and model modules imported successfully.


In [2]:
# Load and align the real processed datasets.
import numpy as np
import pandas as pd

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
required_processed = [
    PROCESSED_DIR / "train_imputed.csv",
    PROCESSED_DIR / "test_imputed.csv",
    PROCESSED_DIR / "test_labels.csv",
]
missing_processed = [path for path in required_processed if not path.exists()]
if missing_processed:
    raise FileNotFoundError(
        "Required processed dataset files are missing: "
        + ", ".join(str(path.relative_to(PROJECT_ROOT)) for path in missing_processed)
        + ". Run notebooks/05_data_preprocessing.ipynb after placing raw "
          "train.csv, test.csv, and test_label.csv in data/raw/."
    )

train_df = pd.read_csv(PROCESSED_DIR / "train_imputed.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test_imputed.csv")
test_label_df = pd.read_csv(PROCESSED_DIR / "test_labels.csv")
timestamp_col = "timestamp_(min)"
label_col = "label"
feature_cols = [column for column in train_df.columns if column.startswith("feature_")]
if not feature_cols:
    raise ValueError("No feature_* columns were found in the processed training data.")
for frame_name, frame in {"train": train_df, "test": test_df, "test labels": test_label_df}.items():
    if timestamp_col not in frame.columns:
        raise ValueError(f"{timestamp_col!r} is missing from {frame_name} data.")
if label_col not in test_label_df.columns:
    raise ValueError(f"{label_col!r} is missing from test labels.")

train_df = train_df.sort_values(timestamp_col).reset_index(drop=True)
test_df = test_df.sort_values(timestamp_col).reset_index(drop=True)
test_label_df = test_label_df.sort_values(timestamp_col).reset_index(drop=True)
if len(test_df) != len(test_label_df):
    raise ValueError("Test features and test labels have different record counts.")
if not np.array_equal(test_df[timestamp_col].to_numpy(), test_label_df[timestamp_col].to_numpy()):
    raise ValueError("Test features and labels are not timestamp-aligned.")

X_train_raw = train_df[feature_cols].copy()
X_test_raw = test_df[feature_cols].copy()
y_test = test_label_df[label_col].astype(int).to_numpy()
if not np.isfinite(X_train_raw.to_numpy(dtype=float)).all() or not np.isfinite(X_test_raw.to_numpy(dtype=float)).all():
    raise ValueError("Feature data contains non-finite values.")
print(f"Training records: {len(X_train_raw):,}; final testing records: {len(X_test_raw):,}")
print(f"Features: {len(feature_cols)}; test labels aligned: {len(X_test_raw) == len(y_test)}")


FileNotFoundError: Required processed dataset files are missing: data\processed\train_imputed.csv, data\processed\test_imputed.csv, data\processed\test_labels.csv. Run notebooks/05_data_preprocessing.ipynb after placing raw train.csv, test.csv, and test_label.csv in data/raw/.

In [ ]:
# Chronological validation split and scaling. No test labels are used here.
from sklearn.preprocessing import StandardScaler

VALIDATION_FRACTION = 0.20
split_index = int(len(X_train_raw) * (1 - VALIDATION_FRACTION))
if split_index <= 0 or split_index >= len(X_train_raw):
    raise ValueError("Training data is too small for the configured validation split.")
X_fit_raw = X_train_raw.iloc[:split_index].reset_index(drop=True)
X_val_raw = X_train_raw.iloc[split_index:].reset_index(drop=True)
fit_timestamps = train_df[timestamp_col].iloc[:split_index].reset_index(drop=True)
val_timestamps = train_df[timestamp_col].iloc[split_index:].reset_index(drop=True)
test_timestamps = test_df[timestamp_col].reset_index(drop=True)
if not (fit_timestamps.iloc[-1] < val_timestamps.iloc[0] < test_timestamps.iloc[0]):
    raise ValueError("Training, validation, and testing periods overlap.")

scaler = StandardScaler()
X_fit_scaled = pd.DataFrame(scaler.fit_transform(X_fit_raw), columns=feature_cols)
X_val_scaled = pd.DataFrame(scaler.transform(X_val_raw), columns=feature_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_raw), columns=feature_cols)
print("Fit / validation / final test:", X_fit_scaled.shape, X_val_scaled.shape, X_test_scaled.shape)


In [ ]:
# Isolation Forest: fit on model-fit data; threshold from unlabeled validation scores.
import time
from sklearn.ensemble import IsolationForest

isolation_model = IsolationForest(n_estimators=200, contamination="auto", random_state=42, n_jobs=-1)
start = time.perf_counter(); isolation_model.fit(X_fit_scaled); isolation_training_time = time.perf_counter() - start
start = time.perf_counter(); if_val_scores = -isolation_model.score_samples(X_val_scaled); isolation_validation_scoring_time = time.perf_counter() - start
IF_THRESHOLD_PERCENTILE = 99.0
if_threshold = float(np.percentile(if_val_scores, IF_THRESHOLD_PERCENTILE))
start = time.perf_counter(); if_test_scores = -isolation_model.score_samples(X_test_scaled); isolation_scoring_time = time.perf_counter() - start
if_test_predictions = (if_test_scores >= if_threshold).astype(int)
print(f"Isolation Forest threshold ({IF_THRESHOLD_PERCENTILE:.1f}th validation percentile): {if_threshold:.6f}")


In [ ]:
# K-Means: use the same validation percentile rule and final test records.
from sklearn.cluster import KMeans

kmeans_model = KMeans(n_clusters=2, random_state=42, n_init=10)
start = time.perf_counter(); kmeans_model.fit(X_fit_scaled); kmeans_training_time = time.perf_counter() - start
start = time.perf_counter(); kmeans_val_scores = np.asarray(calculate_anomaly_scores(kmeans_model, X_val_scaled)); kmeans_validation_scoring_time = time.perf_counter() - start
KMEANS_THRESHOLD_PERCENTILE = IF_THRESHOLD_PERCENTILE
kmeans_threshold = float(np.percentile(kmeans_val_scores, KMEANS_THRESHOLD_PERCENTILE))
start = time.perf_counter(); kmeans_test_scores = np.asarray(calculate_anomaly_scores(kmeans_model, X_test_scaled)); kmeans_scoring_time = time.perf_counter() - start
kmeans_test_predictions = (kmeans_test_scores >= kmeans_threshold).astype(int)
assert len(kmeans_test_scores) == len(if_test_scores) == len(y_test)
print(f"K-Means threshold ({KMEANS_THRESHOLD_PERCENTILE:.1f}th validation percentile): {kmeans_threshold:.6f}")
print("Both models use identical final testing records:", len(y_test))


In [ ]:
# Evaluate both models through src/evaluation.py on the identical final test set.
kmeans_evaluation = evaluate_predictions(y_test, kmeans_test_predictions, kmeans_test_scores)
isolation_evaluation = evaluate_predictions(y_test, if_test_predictions, if_test_scores)
comparison = build_model_comparison([
    {"Model": "K-Means", "Threshold": kmeans_threshold, **kmeans_evaluation["metrics"]},
    {"Model": "Isolation Forest", "Threshold": if_threshold, **isolation_evaluation["metrics"]},
]).set_index("Model")
print(comparison.round(4).to_string())
print("K-Means confusion matrix:\n", kmeans_evaluation["confusion_matrix"])
print("Isolation Forest confusion matrix:\n", isolation_evaluation["confusion_matrix"])
print("Labels and anomaly scores aligned:", len(y_test) == len(kmeans_test_scores) == len(if_test_scores))


In [ ]:
# Measure thresholding and throughput consistently for both models.
def measure_threshold(scores, threshold, runs=20):
    scores = np.asarray(scores)
    _ = (scores >= threshold).astype(int)
    start = time.perf_counter()
    for _ in range(runs):
        predictions = (scores >= threshold).astype(int)
    return predictions, (time.perf_counter() - start) / runs

kmeans_test_predictions, kmeans_threshold_time = measure_threshold(kmeans_test_scores, kmeans_threshold)
if_test_predictions, isolation_threshold_time = measure_threshold(if_test_scores, if_threshold)
kmeans_runtime = calculate_runtime_metrics(kmeans_training_time, kmeans_scoring_time, kmeans_threshold_time, len(y_test))
isolation_runtime = calculate_runtime_metrics(isolation_training_time, isolation_scoring_time, isolation_threshold_time, len(y_test))
runtime_comparison = pd.DataFrame([kmeans_runtime, isolation_runtime], index=["K-Means", "Isolation Forest"])
print(runtime_comparison.round(6).to_string())


In [ ]:
# Save verified CSV files and graphs under the project root.
import matplotlib.pyplot as plt

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(RESULTS_DIR / "final_model_comparison.csv")
runtime_comparison.to_csv(RESULTS_DIR / "runtime_comparison.csv")
pd.DataFrame(kmeans_evaluation["confusion_matrix"], index=["Actual Normal", "Actual Anomaly"], columns=["Predicted Normal", "Predicted Anomaly"]).to_csv(RESULTS_DIR / "kmeans_confusion_matrix.csv")
pd.DataFrame(isolation_evaluation["confusion_matrix"], index=["Actual Normal", "Actual Anomaly"], columns=["Predicted Normal", "Predicted Anomaly"]).to_csv(RESULTS_DIR / "isolation_forest_confusion_matrix.csv")
pd.DataFrame({"Actual_Label": y_test, "KMeans_Prediction": kmeans_test_predictions, "Isolation_Forest_Prediction": if_test_predictions, "KMeans_Score": kmeans_test_scores, "Isolation_Forest_Score": if_test_scores}).to_csv(RESULTS_DIR / "final_test_predictions.csv", index=False)

comparison.drop(columns="Threshold").plot(kind="bar", figsize=(10, 6), ylim=(0, 1))
plt.title("K-Means vs Isolation Forest Performance Comparison"); plt.ylabel("Score"); plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.savefig(FIGURES_DIR / "model_performance_comparison.png", dpi=300); plt.close()
runtime_comparison[["Training Time (s)", "Scoring Time (s)", "Threshold Time (s)", "Total Inference Time (s)"]].plot(kind="bar", figsize=(11, 6))
plt.title("K-Means vs Isolation Forest Runtime Comparison"); plt.ylabel("Seconds"); plt.tight_layout(); plt.savefig(FIGURES_DIR / "runtime_comparison.png", dpi=300); plt.close()
runtime_comparison["Throughput (records/s)"].plot(kind="bar", figsize=(8, 5))
plt.title("Model Throughput Comparison"); plt.ylabel("Records per second"); plt.tight_layout(); plt.savefig(FIGURES_DIR / "throughput_comparison.png", dpi=300); plt.close()
print("Saved evaluation CSV files to:", RESULTS_DIR)
print("Saved graphs to:", FIGURES_DIR)


In [ ]:
# Final integration assertions.
assert len(y_test) == len(kmeans_test_scores) == len(if_test_scores)
assert len(y_test) == len(kmeans_test_predictions) == len(if_test_predictions)
assert IF_THRESHOLD_PERCENTILE == KMEANS_THRESHOLD_PERCENTILE
assert set(comparison.index) == {"K-Means", "Isolation Forest"}
assert set(runtime_comparison.index) == {"K-Means", "Isolation Forest"}
print("Integration checks passed.")
